# Melting Point 예측 — v3_gnn
GNN (Graph Isomorphism Network) | PyTorch Geometric | SMILES → 분자 그래프

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scikit-learn rdkit torch torch-geometric -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.data import Data, DataLoader as GeoLoader
from torch_geometric.nn import GINConv, global_mean_pool, global_max_pool

from rdkit import Chem
from rdkit.Chem import rdchem

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("라이브러리 로드 완료")
print(f"디바이스: {DEVICE}")

## 1. 데이터 로드 및 정리

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/sunghee-sketch/middle-test/main/data/Melting_point_2.csv")
df = df.drop_duplicates().reset_index(drop=True)
df = df[df["MP"] >= 0].reset_index(drop=True)
df["MP_log"] = np.log1p(df["MP"])
print(f"데이터 크기: {df.shape}")
print(f"MP 범위: {df["MP"].min():.1f} ~ {df["MP"].max():.1f} K")
df.head()

## 2. 분자 그래프 변환

In [ ]:
# GNN은 분자를 그래프로 표현 — 피처 엔지니어링 불필요
# 원자(노드) 피처: 원자번호, 결합수, 전하, 고리여부, 방향족여부, 수소수, 혼성화
# 결합(엣지) 피처: 결합 종류(단일/이중/삼중/방향족)

HYBRIDIZATION = {
    rdchem.HybridizationType.SP:  0,
    rdchem.HybridizationType.SP2: 1,
    rdchem.HybridizationType.SP3: 2,
    rdchem.HybridizationType.SP3D:  3,
    rdchem.HybridizationType.SP3D2: 4,
    rdchem.HybridizationType.OTHER: 5,
}

def atom_features(atom):
    return [
        atom.GetAtomicNum() / 100.0,
        atom.GetDegree() / 6.0,
        atom.GetFormalCharge() / 4.0,
        float(atom.IsInRing()),
        float(atom.GetIsAromatic()),
        atom.GetTotalNumHs() / 4.0,
        HYBRIDIZATION.get(atom.GetHybridization(), 5) / 5.0,
    ]

BOND_TYPE = {
    rdchem.BondType.SINGLE:    0,
    rdchem.BondType.DOUBLE:    1,
    rdchem.BondType.TRIPLE:    2,
    rdchem.BondType.AROMATIC:  3,
}

def smiles_to_graph(smi, y_val):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    # 노드 피처
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    # 엣지 (양방향)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = BOND_TYPE.get(bond.GetBondType(), 0)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [[bt / 3.0], [bt / 3.0]]
    if not edge_index:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([y_val], dtype=torch.float))

print("분자 그래프 변환 중...")
graph_list, valid_idx = [], []
for i, (smi, y_log) in enumerate(zip(df["SMILES"], df["MP_log"])):
    g = smiles_to_graph(smi, y_log)
    if g is not None:
        graph_list.append(g)
        valid_idx.append(i)

y_orig = df["MP"].values[valid_idx]
print(f"유효 그래프: {len(graph_list)} / {len(df)}")
print(f"노드 피처 차원: {graph_list[0].x.shape[1]}")
print(f"예시 그래프 — 원자수: {graph_list[0].x.shape[0]}, 결합수: {graph_list[0].edge_index.shape[1]//2}")

## 3. Train/Test 분할

In [ ]:
mp_bins = pd.qcut(y_orig, q=10, labels=False, duplicates="drop")
indices = np.arange(len(graph_list))
tr_idx, te_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_STATE, stratify=mp_bins
)
train_graphs = [graph_list[i] for i in tr_idx]
test_graphs  = [graph_list[i] for i in te_idx]
y_train = y_orig[tr_idx]
y_test  = y_orig[te_idx]
print(f"Train: {len(train_graphs)}개  |  Test: {len(test_graphs)}개")

## 4. GNN 모델 정의 (GIN)

In [ ]:
# GIN = Graph Isomorphism Network
# 분자 그래프에서 원자 간 정보를 message passing으로 전달
# Global Pooling으로 분자 전체 표현 생성 → MLP Head로 MP 예측

class GINModel(nn.Module):
    def __init__(self, node_dim=7, hidden_dim=128, num_layers=4):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        in_dim = node_dim
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            in_dim = hidden_dim
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
        # Global mean + max pooling 결합
        x = torch.cat([global_mean_pool(x, batch),
                        global_max_pool(x, batch)], dim=1)
        return self.head(x).squeeze(1)

print("GIN 모델 구조:")
print("SMILES → 분자 그래프 → GINConv × 4층 → Global Pooling → MLP Head → MP 예측")
test_model = GINModel()
total_params = sum(p.numel() for p in test_model.parameters())
print(f"총 파라미터 수: {total_params:,}")

In [ ]:
def train_epoch_gnn(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = F.mse_loss(pred, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def predict_gnn(model, graphs, batch_size=64):
    model.eval()
    loader = GeoLoader(graphs, batch_size=batch_size)
    preds = []
    for batch in loader:
        batch = batch.to(DEVICE)
        preds.append(model(batch.x, batch.edge_index, batch.batch).cpu().numpy())
    return np.concatenate(preds)

print("학습 함수 정의 완료")

## 5. 5-Fold Cross Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
mp_bins_train = pd.qcut(y_train, q=10, labels=False, duplicates="drop")
cv_r2, cv_mse, cv_mae = [], [], []

EPOCHS   = 100
BATCH    = 32
LR       = 1e-3
PATIENCE = 15

train_indices = np.arange(len(train_graphs))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_indices, mp_bins_train), 1):
    tr_graphs  = [train_graphs[i] for i in tr_idx]
    val_graphs = [train_graphs[i] for i in val_idx]
    y_val_orig = y_train[val_idx]

    tr_ldr = GeoLoader(tr_graphs, batch_size=BATCH, shuffle=True)

    model     = GINModel().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5)

    best_val_loss, patience_cnt, best_state = np.inf, 0, None
    for epoch in range(EPOCHS):
        train_epoch_gnn(model, tr_ldr, optimizer)
        val_pred_log = predict_gnn(model, val_graphs)
        val_loss = mean_squared_error(
            [g.y.item() for g in val_graphs], val_pred_log)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            break

    model.load_state_dict(best_state)
    pred = np.expm1(predict_gnn(model, val_graphs))
    cv_r2.append(r2_score(y_val_orig, pred))
    cv_mse.append(mean_squared_error(y_val_orig, pred))
    cv_mae.append(mean_absolute_error(y_val_orig, pred))
    print(f"Fold {fold}  R²={cv_r2[-1]:.4f}  MSE={cv_mse[-1]:.2f}  MAE={cv_mae[-1]:.2f}  (epoch {epoch+1})")

print()
print("=== 5-Fold CV 평균 ===")
print(f"R²  : {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}")
print(f"MSE : {np.mean(cv_mse):.2f} ± {np.std(cv_mse):.2f}")
print(f"MAE : {np.mean(cv_mae):.2f} ± {np.std(cv_mae):.2f}")

## 6. 최종 모델 — Test 평가

In [ ]:
val_size = int(len(train_graphs) * 0.1)
final_tr  = train_graphs[val_size:]
final_val = train_graphs[:val_size]

final_model = GINModel().to(DEVICE)
optimizer   = optim.Adam(final_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5)
ldr_final   = GeoLoader(final_tr, batch_size=BATCH, shuffle=True)

best_val_loss, patience_cnt, best_state = np.inf, 0, None
for epoch in range(200):
    train_epoch_gnn(final_model, ldr_final, optimizer)
    vl = mean_squared_error(
        [g.y.item() for g in final_val], predict_gnn(final_model, final_val))
    scheduler.step(vl)
    if vl < best_val_loss:
        best_val_loss, patience_cnt = vl, 0
        best_state = {k: v.clone() for k, v in final_model.state_dict().items()}
    else:
        patience_cnt += 1
    if patience_cnt >= PATIENCE:
        break

final_model.load_state_dict(best_state)
y_pred = np.expm1(predict_gnn(final_model, test_graphs))

test_r2  = r2_score(y_test, y_pred)
test_mse = mean_squared_error(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)
print("=== Test Set 성능 ===")
print(f"R²  : {test_r2:.4f}")
print(f"MSE : {test_mse:.2f}")
print(f"MAE : {test_mae:.2f}")

## 7. 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.4, s=15, color="steelblue")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect")
ax.set_xlabel("실제 MP (K)")
ax.set_ylabel("예측 MP (K)")
ax.set_title(f"Predicted vs Actual  (R²={test_r2:.3f})")
ax.legend()

ax = axes[1]
residuals = y_pred - y_test
ax.hist(residuals, bins=40, color="salmon", edgecolor="white")
ax.axvline(0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("잔차 (예측 − 실제)")
ax.set_ylabel("Count")
ax.set_title("잔차 분포")

ax = axes[2]
ax.bar(range(1, 6), cv_r2, color="mediumseagreen", edgecolor="white")
ax.axhline(np.mean(cv_r2), color="red", linestyle="--", label=f"Mean={np.mean(cv_r2):.3f}")
ax.set_xlabel("Fold")
ax.set_ylabel("R²")
ax.set_title("5-Fold CV R² per Fold")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.show()

## 8. 결과 요약

In [ ]:
summary = pd.DataFrame({
    "구분":  ["5-Fold CV 평균", "Test Set"],
    "R²":   [f"{np.mean(cv_r2):.4f}", f"{test_r2:.4f}"],
    "MSE":  [f"{np.mean(cv_mse):.2f}", f"{test_mse:.2f}"],
    "MAE":  [f"{np.mean(cv_mae):.2f}", f"{test_mae:.2f}"],
})
print("=== 버전별 비교 ===")
print("base  CV R²: 0.5346  Test R²: 0.4831")
print(f"v3gnn CV R²: {np.mean(cv_r2):.4f}  Test R²: {test_r2:.4f}")
summary